# Convolutional Neural Networks

Humne pehle dekha hai ki neural networks images ke saath kaam karne mein kaafi achhe hote hain, aur ek-layer perceptron bhi MNIST dataset ke handwritten digits ko reasonable accuracy ke saath pehchaan leta hai. Lekin MNIST dataset kaafi special hai, aur saare digits image ke andar centered hote hain, jisse yeh task simple ho jaata hai.

Real life mein hum chahenge ki image mein objects ko unki exact position ki parwah kiye bina pehchaan sakein. Computer vision general classification se alag hai, kyunki jab hum image mein koi particular object dhoondh rahe hote hain, toh hum image ko scan karte hain aur kuch khaas **patterns** aur unke combinations dhoondhte hain. Jaise, jab hum billi dhoondh rahe hote hain, toh pehle horizontal lines dekh sakte hain, jo whiskers bana sakti hain, aur phir whiskers ke kuch khaas combinations humein bata sakte hain ki yeh sach mein ek billi ki tasveer hai. Kuch patterns ki relative position aur presence important hoti hai, na ki image mein unki exact position.

Patterns extract karne ke liye hum **convolutional filters** ka concept use karenge. Lekin pehle, chalo woh saari dependencies aur functions load kar lete hain jo humne pichhle units mein define kiye the. Hum `tfcv` helper library bhi import karenge, jisme kuch useful functions hain jinhe hum is notebook ke andar define nahi karna chahte, taaki code chhota aur saaf-suthra rahe.

In [ ]:
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import numpy as np
from tfcv import *

Is example mein hum us MNIST dataset par focus karenge jo humne pehle dekha hai, aur image classification par. Hum Keras ke built-in functions use karke dataset load karne se shuru karenge.

In [ ]:
(x_train,y_train),(x_test,y_test) = keras.datasets.mnist.load_data()
x_train = x_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0

## Convolutional Filters

Convolutional filters chhoti windows hote hain jo image ke har pixel par chalte hain aur neighboring pixels ka weighted average calculate karte hain.



Yeh weight coefficients ke matrix se define hote hain. Chalo dekhte hain hamare MNIST handwritten digits par do alag-alag convolutional filters apply karne ke examples:

In [ ]:
plot_convolution(x_train[:5],[[-1.,0.,1.],[-1.,0.,1.],[-1.,0.,1.]],'Vertical edge filter')
plot_convolution(x_train[:5],[[-1.,-1.,-1.],[0.,0.,0.],[1.,1.,1.]],'Horizontal edge filter')

Pehla filter **vertical edge filter** kehlata hai, aur ise neeche diye gaye matrix se define kiya gaya hai:
$$
\left(
    \begin{matrix}
     -1 & 0 & 1 \cr
     -1 & 0 & 1 \cr
     -1 & 0 & 1 \cr
    \end{matrix}
\right)
$$
Jab yeh filter relatively uniform pixel region par chalta hai, toh saari values jodkar 0 ho jaati hain. Lekin jab yeh image mein vertical edge se takraata hai, toh ek high spike value produce hoti hai. Isi wajah se upar di gayi images mein aap dekh sakte hain ki vertical edges high aur low values se represent hote hain, jabki horizontal edges average ho jaate hain.

Iske ulta tab hota hai jab hum horizontal edge filter apply karte hain - horizontal lines badh jaati hain, aur vertical lines average ho jaati hain.

Classical computer vision mein image par kai filters apply kiye jaate the taaki features generate hon, jinhe phir machine learning algorithms classifier banane ke liye use karte the. Yeh filters asal mein un neural structures jaise hain jo kuch janwaron ke vision system mein hote hain.

<img src="images/lmfilters.jpg" width="400"/>

Lekin deep learning mein hum aise networks banate hain jo classification problem solve karne ke liye **best convolutional filters** khud seekhte hain. Iske liye hum **convolutional layers** introduce karte hain.

## Convolutional Layers

Convolutional layer ke weights ko trainable banane ke liye, humein kisi tarah image par convolutional filter window apply karne ki process ko matrix operations mein badalna hoga, jise phir backward propagation training ke liye use kiya ja sake. Yeh karne ke liye hum ek smart matrix transformation use karte hain, jise hum **im2col** kehte hain.

Maan lijiye hamare paas ek chhoti image $\mathbf{x}$ hai, jisme yeh pixels hain:

$$
\mathbf{x} = \left(
         \begin{array}{ccccc}
           a & b & c & d & e \\
           f & g & h & i & j \\
           k & l & m & n & o \\
           p & q & r & s & t \\
           u & v & w & x & y \\
         \end{array}
     \right)
$$

Aur hum do convolutional filters apply karna chahte hain, jinke weights yeh hain:
$$
W^{(i)} = \left(\begin{array}{ccc}
            w^{(i)}_{00} & w^{(i)}_{01} & w^{(i)}_{02} \\
            w^{(i)}_{10} & w^{(i)}_{11} & w^{(i)}_{12} \\
            w^{(i)}_{20} & w^{(i)}_{21} & w^{(i)}_{22} \\
            \end{array}\right)
$$

Jab convolution apply hota hai, toh result ka pehla pixel
$\left(\begin{array}{ccc}
  a & b & c \\
  f & g & h \\
  k & l & m \\
\end{array}\right)$ aur $W^{(i)}$ ke element-wise multiplication se milta hai. Doosra element
$\left(\begin{array}{ccc}
  b & c & d \\
  g & h & i \\
  l & m & n \\
\end{array}\right)$ aur $W^{(i)}$ ke multiplication se milta hai, aur isi tarah aage.

Is process ko formalize karne ke liye, chalo original image $x$ ke saare $3\times3$ fragments ko neeche diye gaye matrix mein extract karte hain:

$$
\mathrm{im2col}(x) = \left[
        \begin{array}{cccccc}
          a & b & \ldots & g & \ldots & m \\
          b & c & \ldots & h & \ldots & n \\
          c & d & \ldots & i & \ldots & o \\
          f & g & \ldots & l & \ldots & r \\
          g & h & \ldots & m & \ldots & s \\
          h & i & \ldots & n & \ldots & t \\
          k & l & \ldots & q & \ldots & w \\
          l & m & \ldots & r & \ldots & x \\
          m & n & \ldots & s & \ldots & y \\
        \end{array}
    \right]
$$

Is matrix ka har column original image ke har $3\times3$ subregion se match karta hai. Ab convolution ka result paane ke liye humein bas is matrix ko weights ke matrix se multiply karna hai:
$$
\mathbf{W} = \left[
         \begin{array}{cccccccc}
            w^{(0)}_{00} & w^{(0)}_{01} & w^{(0)}_{02} & w^{(0)}_{10} & w^{(0)}_{11} & \ldots & w^{(0)}_{21} & w^{(0)}_{22} \\
            w^{(1)}_{00} & w^{(1)}_{01} & w^{(1)}_{02} & w^{(1)}_{10} & w^{(1)}_{11} & \ldots & w^{(1)}_{21} & w^{(1)}_{22} \\
         \end{array}
       \right]
$$
(is matrix ki har row $i$-th filter ke weights ko flatten karke ek row mein rakhti hai)

Is tarah, original image par convolutional filter apply karne ki process ko matrix multiplication mein badla ja sakta hai, jise hum backpropagation ke saath handle karna pehle se jaante hain:
$$
C(x) = W\times\mathbf{im2col}(x)
$$

Convolutional layers `Conv2d` class se define hoti hain. Humein yeh specify karna hota hai:
* `filters` - use karne wale filters ki ginti. Hum 9 alag-alag filters use karenge, jo network ko yeh pata lagane ka kaafi mauka denge ki hamare scenario ke liye kaun se filters sabse achhe hain.
* `kernel_size` - sliding window ka size. Aam taur par 3x3 ya 5x5 filters use hote hain.

Sabse simple CNN mein ek convolutional layer hogi. 28x28 input size ko dekhte hue, nau 5x5 filters apply karne ke baad hamare paas 24x24x9 ka tensor hoga. Spatial dimension chhota ho jaata hai, kyunki sirf 24 positions hain jahan 5 length ka sliding interval 28 pixels mein fit ho sakta hai.

Convolution ke baad, hum 24x24x9 tensor ko ek single vector mein flatten kar dete hain, jiska size 5184 hota hai, aur phir ek linear layer add karte hain, jo 10 classes produce karti hai. Hum layers ke beech `relu` activation function bhi use karte hain.

In [ ]:
model = keras.models.Sequential([
    keras.layers.Conv2D(filters=9, kernel_size=(5,5), input_shape=(28,28,1),activation='relu'),
    keras.layers.Flatten(),
    keras.layers.Dense(10)
])

model.compile(loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),metrics=['acc'])

model.summary()

Aap dekh sakte hain ki is network mein lagbhag 50k trainable parameters hain, jabki fully connected multi-layered networks mein lagbhag 80k hote hain. Isse hum chhote datasets par bhi achhe results paa sakte hain, kyunki convolutional networks generalization mein kaafi behtar hote hain.

> **Note**: Zyadatar practical cases mein hum colored images par convolutional layers apply karna chahte hain. Isliye `Conv2D` layer input ko $W\times H\times C$ shape mein expect karti hai, jahan $W$ aur $H$ image ki width aur height hain, aur $C$ color channels ki ginti hai. Grayscale images ke liye humein wahi shape chahiye jisme $C=1$ ho.

Training shuru karne se pehle humein apne data ko reshape karna hoga:

In [ ]:
x_train_c = np.expand_dims(x_train,3)
x_test_c = np.expand_dims(x_test,3)
hist = model.fit(x_train_c,y_train,validation_data=(x_test_c,y_test),epochs=5)

In [ ]:
plot_results(hist)

Jaisa ki aap dekh sakte hain, hum pehle unit ke fully connected networks ke mukable zyada accuracy aur tezi (epochs ki ginti ke hisaab se) hasil kar paa rahe hain. Lekin training ke liye zyada resources chahiye, aur yeh non-GPU computers par slow ho sakti hai.

## Convolutional layers ko visualize karna

Hum apni trained convolutional layers ke weights ko bhi visualize kar sakte hain, taaki samajhne ki koshish kar sakein ki asal mein kya ho raha hai:

In [ ]:
fig,ax = plt.subplots(1,9)
l = model.layers[0].weights[0]
for i in range(9):
    ax[i].imshow(l[...,0,i])
    ax[i].axis('off')

Aap dekh sakte hain ki kuch filters diagonal lines pehchaanne mein capable lagte hain, jabki baaki kaafi random dikhte hain.

> **Task**: Wahi network 3x3 filters ke saath train karein aur unhe visualize karein. Kya aapko zyada familiar patterns dikhte hain?

## Multi-layered CNNs aur pooling layers

Pehli convolutional layer primary patterns dhoondhti hai, jaise horizontal ya vertical lines, lekin hum unke upar aur convolutional layers laga sakte hain taaki higher-level patterns, jaise primary shapes, dhoondhe ja sakein. Phir aur convolutional layers un shapes ko image ke kuch hisson mein jod sakti hain, aur aakhir mein us final object tak pahunch sakti hain jise hum classify karne ki koshish kar rahe hain.

Aisa karte waqt hum ek aur trick bhi laga sakte hain: image ka spatial size kam karna. Ek baar jab humne sliding 3x3 window ke andar horizontal line detect kar li, toh yeh utna important nahi rehta ki woh exactly kis pixel par hui. Isliye hum image ke size ko "scale down" kar sakte hain, jo **pooling layers** use karke kiya jaata hai:

 * **Average pooling** ek sliding window (jaise 2x2 pixels) leti hai aur window ke andar ki values ka average nikaalti hai.
 * **Max pooling** window ko maximum value se replace kar deti hai. Max pooling ke peeche idea yeh hai ki sliding window ke andar kisi particular pattern ki presence detect ki jaaye.

Is tarah, ek typical CNN mein kai convolutional layers hoti hain, jinke beech pooling layers hoti hain taaki image ke dimensions kam ho sakein. Hum filters ki ginti bhi badhate hain, kyunki jaise-jaise patterns advanced hote jaate hain - humein dekhne ke liye zyada possible interesting combinations hote hain.

![Convolutional layers aur pooling layers dikhane wali ek image.](../../../../../lessons/4-ComputerVision/07-ConvNets/images/cnn-pyramid.png)

Spatial dimensions kam karne aur feature/filter dimensions badhane ki wajah se is architecture ko **pyramid architecture** bhi kehte hain.

In [ ]:
model = keras.models.Sequential([
    keras.layers.Conv2D(filters=10, kernel_size=(5,5), input_shape=(28,28,1),activation='relu'),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(filters=20, kernel_size=(5,5), activation='relu'),
    keras.layers.MaxPooling2D(),    
    keras.layers.Flatten(),
    keras.layers.Dense(10)
])

model.compile(loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),metrics=['acc'])

model.summary()

Trainable parameters ki ginti (~8.5K) pehle ke cases ke mukable dramatically kam hai. Aisa isliye hota hai kyunki aam taur par convolutional layers mein parameters kam hote hain, aur aakhri dense layer apply karne se pehle image ki dimensionality kaafi had tak kam ho jaati hai. Parameters ki kam ginti ka hamare models par positive effect padta hai, kyunki yeh chhote dataset size par bhi overfitting rokne mein madad karta hai.

In [ ]:
hist = model.fit(x_train_c,y_train,validation_data=(x_test_c,y_test),epochs=5)

In [ ]:
plot_results(hist)

Aap dekh sakte hain ki hum sirf ek layer ke mukable zyada accuracy paa rahe hain, aur woh bhi bahut tezi se, sirf 1 ya 2 epochs mein. Iska matlab hai ki complex network architectures ko yeh samajhne ke liye bahut kam data chahiye ki kya ho raha hai, aur hamari images se general patterns nikaalne ke liye. Lekin training mein zyada time lagta hai aur GPU ki zaroorat padti hai.

## CIFAR-10 dataset ki real images ke saath experiment karna

Halaanki hamare handwritten digits pehchaanne ki problem ek simple problem lag sakti hai, ab hum kuch zyada serious karne ke liye ready hain. Chalo alag-alag objects ki tasveeron ke ek advanced dataset ko explore karte hain, jise [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) kehte hain. Isme 60,000 32x32 images hain, jo 10 classes mein baanti gayi hain.

In [ ]:
(x_train,y_train),(x_test,y_test) = keras.datasets.cifar10.load_data()
x_train = x_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0
classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [ ]:
display_dataset(x_train,y_train,classes=classes)

In [ ]:
model = keras.models.Sequential([
    keras.layers.Conv2D(filters = 6, kernel_size = 5, strides = 1, activation = 'relu', input_shape = (32,32,3)),
    keras.layers.MaxPooling2D(pool_size = 2, strides = 2),
    keras.layers.Conv2D(filters = 16, kernel_size = 5, strides = 1, activation = 'relu'),
    keras.layers.MaxPooling2D(pool_size = 2, strides = 2),
    keras.layers.Flatten(),
    keras.layers.Dense(120, activation = 'relu'),
    keras.layers.Dense(84, activation = 'relu'),
    keras.layers.Dense(10, activation = 'softmax')])

model.summary()

In [ ]:
model.compile(optimizer = 'adam', loss = 'sparse_categorical_crossentropy', metrics = ['acc'])
hist = model.fit(x_train,y_train,validation_data=(x_test,y_test),epochs=10)

In [ ]:
plot_results(hist)

Kuch epochs ki training ke baad humne jo accuracy hasil ki hai, woh bahut shaandaar nahi lagti. Lekin yaad rakhiye ki sirf andaaza lagane se humein 10% accuracy hi milegi, aur hamari problem asal mein MNIST digits ki classification se kaafi mushkil hai. Itne kam training time mein 50% se zyada accuracy paana ek achhi uplabdhi lagti hai.

## Key takeaways

Is unit mein humne computer vision neural networks ke peeche ka main concept seekha - convolutional networks. Real-life architectures, jo image classification, object detection, aur even image generation networks ko power karte hain, sab CNNs par based hain, bas zyada layers aur kuch extra training techniques ke saath.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->